# Balance de deforestacion y recuperacion — bosque manejado de Quebec

Este notebook extrae composites anuales de Sentinel-2 sobre un area del bosque publico bajo aprovechamiento en Quebec, detecta cambio ano contra ano (dNBR) y calcula el balance neto de superficie perdida vs recuperada por unidad espacial.

**Flujo:** extraccion GEE -> composites anuales NDVI/NBR -> deteccion de cambio -> agregacion espacial -> balance neto.

In [ ]:
import sys
sys.path.append('../src')

import ee
import geemap
import pandas as pd

from gee_utils import init_ee, build_annual_stack
from change_detection import build_change_series, aggregate_by_units

PROJECT = "your-gee-project-id"
init_ee(PROJECT)

## 1. Area de estudio: region administrativa Abitibi-Temiscamingue

Fuente: capa de decoupages administratifs de Donnees Quebec (region 08), o alternativamente
la union de las 5 divisions de recensement de Statistique Canada que componen la region
(Temiscamingue, Rouyn-Noranda, Abitibi-Ouest, Abitibi, La Vallee-de-l'Or).

Descargar el shapefile localmente y ajustar la ruta en `SHP_PATH`.

In [ ]:
import geopandas as gpd
import geemap

SHP_PATH = "../data/regions_administratives.shp"  # ajustar segun descarga
REGION_NAME_FIELD = "RES_NM_REG"  # nombre de campo puede variar segun la version del shapefile
REGION_NAME = "Abitibi-T\u00e9miscamingue"

gdf = gpd.read_file(SHP_PATH)
gdf_region = gdf[gdf[REGION_NAME_FIELD].str.contains("Abitibi", case=False, na=False)]
gdf_region = gdf_region.to_crs("EPSG:4326")  # GEE espera WGS84

aoi_fc = geemap.geopandas_to_ee(gdf_region)
AOI = aoi_fc.geometry()

Map = geemap.Map()
Map.centerObject(AOI, 8)
Map.addLayer(AOI, {}, "AOI - Abitibi-Temiscamingue")
Map

## 2. Composites anuales (NDVI/NBR)

In [ ]:
YEARS = range(2017, 2026)
annual_stack = build_annual_stack(YEARS, AOI)
print(f"Composites generados: {annual_stack.size().getInfo()}")

In [ ]:
# Visualizacion rapida del ultimo composite (NBR)
last_img = ee.Image(annual_stack.sort('year', False).first())
nbr_vis = {"min": -0.5, "max": 0.8, "palette": ["red", "white", "green"]}

Map2 = geemap.Map()
Map2.centerObject(AOI, 9)
Map2.addLayer(last_img.select('NBR'), nbr_vis, "NBR ultimo anio")
Map2

## 3. Deteccion de cambio ano contra ano

In [ ]:
change_pairs = build_change_series(annual_stack, band="NBR")
print([year for year, _ in change_pairs])

## 4. Agregacion espacial y balance neto

Requiere una capa de unidades espaciales (`units_fc`) con propiedad `unit_id` — por ejemplo una grilla de hexagonos o la subdivision de la UAF. Placeholder: generar grilla con `geemap` o subir asset propio.

In [ ]:
# Placeholder: reemplazar por la capa real de unidades espaciales
# units_fc = ee.FeatureCollection("projects/your-gee-project-id/assets/hex_grid")

results = []
for year, classified in change_pairs:
    # stats = aggregate_by_units(classified, units_fc, scale=10)
    # df_year = geemap.ee_to_df(stats)
    # df_year['year'] = year
    # results.append(df_year)
    pass

# balance_df = pd.concat(results, ignore_index=True)
# balance_df.to_csv('../figures/net_balance_by_unit.csv', index=False)

## 5. Resultado: balance neto acumulado

Grafico final para el post de LinkedIn: superficie perdida vs recuperada por anio, y balance neto acumulado.

In [ ]:
import matplotlib.pyplot as plt

# balance_summary = balance_df.groupby('year')[['loss_ha', 'recovery_ha', 'net_balance_ha']].sum()
# balance_summary['net_balance_cumsum'] = balance_summary['net_balance_ha'].cumsum()

# fig, ax = plt.subplots(figsize=(9, 5))
# balance_summary[['loss_ha', 'recovery_ha']].plot(kind='bar', ax=ax)
# ax.set_ylabel('Hectareas')
# ax.set_title('Perdida vs recuperacion anual — bosque manejado de Quebec')
# plt.tight_layout()
# plt.savefig('../figures/annual_loss_recovery.png', dpi=200)
# plt.show()